# MSG data

- Download required MSG data from a product list
- Add radiation data to an existing dataset

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os

import geopandas as gpd
import pandas as pd
import rasterio as rio
import xarray as xr
from pyproj import CRS

In [ ]:
from etdataset import msg
from etdataset.interpolation import create_grid

## Inputs

In [ ]:
# Read the CSV into a pandas DataFrame
df = pd.read_csv("landsat_products.csv")

df["footprint"] = df["geometry"]
# Convert the WKT column to geometry
df["geometry"] = gpd.GeoSeries.from_wkt(df["overlap_geometry"])

# Now create a GeoDataFrame
product_list = gpd.GeoDataFrame(df, geometry="geometry", crs=CRS(4326))

In [ ]:
product_list

In [ ]:
product_list.geometry.bounds

## Download required data

In [ ]:
date = dt.datetime(2024, 10, 21, 11, 10)

Download daily DSSF

In [ ]:
msg.download_msg_file(
    satellite=msg.MSGSatellite.MSG,
    product=msg.MSGProduct.DAILY_SURFACE_SOLAR_RADIATION_DOWNWARD,
    date=date,
    fmt=msg.MSGFormat.NETCDF,
    path=os.getcwd(),
)

In [ ]:
msg.download_msg_file(
    satellite=msg.MSGSatellite.MSGIODC,
    product=msg.MSGProduct.DAILY_SURFACE_SOLAR_RADIATION_DOWNWARD,
    date=date,
    fmt=msg.MSGFormat.NETCDF,
    path=os.getcwd(),
)

Download DSSF

In [ ]:
msg.download_msg_file(
    satellite=msg.MSGSatellite.MSG,
    product=msg.MSGProduct.SURFACE_SOLAR_RADIATION_DOWNWARD,
    date=date,
    fmt=msg.MSGFormat.NETCDF,
    path=os.getcwd(),
)

### Read data 

In [ ]:
data = xr.open_dataset("NETCDF4_LSASAF_MSG_DIDSSF_MSG-Disk_202410210000.nc")

In [ ]:
data

In [ ]:
data.DSSF.plot()

In [ ]:
data = data.set_coords("crs")

### Project on a ROI

In [ ]:
bounds = rio.coords.BoundingBox(*product_list.loc[0].geometry.bounds)
crs = product_list.crs

In [ ]:
grid = create_grid(bounds, crs, 0.1)

In [ ]:
grid.rio.write_crs(crs, inplace=True)